In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
#from imblearn.over_sampling import RandomOverSampler
from sklearn.ensemble import RandomForestClassifier
from interpret import show
from sklearn.metrics import f1_score, accuracy_score

D:\Users\cccc\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
data = pd.read_csv("Datasets/healthcare-dataset-stroke-data.csv")

In [4]:
# One-hot encode all categorical columns
categorical_cols = ["gender","ever_married","work_type","Residence_type","smoking_status"]
encoded = pd.get_dummies(data[categorical_cols], prefix=categorical_cols)

In [7]:
encoded = encoded.astype(int)

In [8]:
# Update data with new columns
data = pd.concat([encoded, data], axis=1)
data.drop(categorical_cols, axis=1, inplace=True)

In [10]:
# Impute missing values of BMI
data.bmi = data.bmi.fillna(0)

In [11]:
# Drop id as it is not relevant
data.drop(["id"], axis=1, inplace=True)

In [12]:
X = data.iloc[:,:-1]
y = data.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=2021)

In [13]:
print(X_train.shape)
print(X_test.shape)

(4088, 21)
(1022, 21)


In [15]:
# %% Fit blackbox model
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(f"F1 Score {f1_score(y_test, y_pred, average='macro')}")
print(f"Accuracy {accuracy_score(y_test, y_pred)}")

F1 Score 0.5244742229666852
Accuracy 0.9510763209393346


In [16]:
import dice_ml

In [35]:
# 1. Split dtypes correctly
continuous_features = ['age', 'avg_glucose_level', 'bmi']
categorical_features = [c for c in X.columns if c not in continuous_features]

X[continuous_features] = X[continuous_features].astype(float)
X[categorical_features] = X[categorical_features].astype(int)

X_test[continuous_features] = X_test[continuous_features].astype(float)
X_test[categorical_features] = X_test[categorical_features].astype(int)

# 2. Rebuild DiCE objects
d = dice_ml.Data(dataframe=pd.concat([X, y], axis=1), 
                 continuous_features=continuous_features,
                 outcome_name='stroke') # your target column name

In [36]:
# Model
rf_dice = dice_ml.Model(model=rf, 
                        # There exist backends for tf, torch, ...
                        backend="sklearn")

In [37]:
explainer = dice_ml.Dice(data_dice, 
                         rf_dice, 
                         # Random sampling, genetic algorithm, kd-tree,...
                         method="genetic")

In [40]:
# %% Create explanation
# Generate CF based on the blackbox model
input_datapoint = X_test[0:1]
cf = explainer.generate_counterfactuals(input_datapoint, 
                                  total_CFs=3, 
                                  desired_class="opposite")

100%|██████████| 1/1 [00:01<00:00,  1.33s/it]


In [41]:
# Visualize it
cf.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 0)


,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,...,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
0,0,1,0,0,1,0,0,1,0,0,...,0,1,0,0,70.0,0,0,72.559998,30.4,0



Diverse Counterfactual set (new outcome: 1)


,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,...,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
0,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,78.0,-,-,78.03,23.9,1.0
0,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,66.0,-,-,76.46,21.2,1.0
0,1.0,0.0,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,1.0,59.35,32.3,1.0


In [42]:
# %% Create feasible (conditional) Counterfactuals
features_to_vary=['avg_glucose_level',
                  'bmi',
                  'smoking_status_smokes']
permitted_range={'avg_glucose_level':[50,250],
                'bmi':[18, 35]}

In [43]:
# Now generating explanations using the new feature weights
cf = explainer.generate_counterfactuals(input_datapoint, 
                                  total_CFs=3, 
                                  desired_class="opposite",
                                  permitted_range=permitted_range,
                                  features_to_vary=features_to_vary)

100%|██████████| 1/1 [00:35<00:00, 35.75s/it]


In [44]:
# Visualize it
cf.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 0)


,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,...,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
0,0,1,0,0,1,0,0,1,0,0,...,0,1,0,0,70.0,0,0,72.559998,30.4,0



Diverse Counterfactual set (new outcome: 1)


,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,...,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
0,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,76.46,21.2,1.0
